# Redox FHIR Pipeline - Drop Tables (Full Refresh)

This utility notebook drops all streaming tables to enable a full refresh of the pipeline.

## Warning
**This will delete all data in the streaming tables.** Only run this when you need to:
- Perform a complete re-ingestion
- Reset the pipeline state
- Handle major schema changes

_Note: Attach to a Serverless SQL Warehouse for execution._

In [ ]:
-- ============================================================================
-- CONFIGURATION
-- ============================================================================
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT 'redox_fhir';
DECLARE OR REPLACE VARIABLE schema_use STRING DEFAULT 'bronze';

SET VARIABLE catalog_use = COALESCE(:catalog_use, catalog_use);
SET VARIABLE schema_use = COALESCE(:schema_use, schema_use);

USE IDENTIFIER(catalog_use || '.' || schema_use);
SELECT current_catalog() AS catalog, current_schema() AS schema;

In [ ]:
-- List all tables before dropping
SHOW TABLES;

## Drop Bronze Tables

In [ ]:
-- Drop bronze layer tables
DROP TABLE IF EXISTS fhir_bronze;
DROP TABLE IF EXISTS fhir_bronze_variant;
DROP TABLE IF EXISTS bundle_meta;
DROP TABLE IF EXISTS resources_exploded;
DROP TABLE IF EXISTS resource_schemas;

## Drop Silver Tables (Dynamic)

This cell dynamically drops all Silver resource tables.

In [ ]:
%python
# Dynamically drop all tables in the schema (except system tables)
tables_df = spark.sql("SHOW TABLES")

# Core bronze tables to skip (already dropped above)
bronze_tables = {
    'fhir_bronze', 
    'fhir_bronze_variant', 
    'bundle_meta', 
    'resources_exploded', 
    'resource_schemas'
}

for row in tables_df.collect():
    table_name = row.tableName
    if table_name.lower() not in bronze_tables:
        print(f"Dropping table: {table_name}")
        spark.sql(f"DROP TABLE IF EXISTS {table_name}")

print("\nAll tables dropped successfully.")

In [ ]:
-- Verify all tables are dropped
SHOW TABLES;